# **MACRO ANALYSIS AND OPPORTUNITIES FOR MED TECH INDUSTRY EUROPE**

The following work was made to analyze the opportunities in the European Market for companies in the Medtech field.

**Source of Information**:

1) Company Reports: We get the financial reports of the company for the last three years.
2) CORDIS - EU research projects under HORIZON EUROPE (2021 - 2027)
3) Clinical Trias Information from *"https://clinicaltrials.gov/api/v2/studies"*
4) US information used for comparison purpose: *"https://api.reporter.nih.gov/v2/projects/search"*

## 1. CORDIS - EU RESEARCH PROJECTS UNDER HORIZON EUROPE

### Import Library and Get the Information from web

In [ ]:
import sys
!{sys.executable} -m pip install openpyxl

In [ ]:
import zipfile
import requests
import urllib.request
import pandas as pd
from pathlib import Path
import os
os.getcwd()

In [ ]:
url="https://cordis.europa.eu/data/cordis-HORIZONprojects-xlsx.zip"
reports_dir= Path("data/Reports")
reports_dir.mkdir(parents=True, exist_ok=True)
zip_path = reports_dir / "cordis-HORIZONprojects-xlsx.zip"
urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extract("project.xlsx",reports_dir)
    z.extract("organization.xlsx",reports_dir)



In [ ]:
df_eu_org=pd.read_excel(report_dir/"organization.xlsx", engine='openpyxl')
df_eu_proj=pd.read_excel(report_dir/"project.xlsx", engine='openpyxl')

In [ ]:
df_eu_org.head()

In [ ]:
df_eu_proj.head()

In [ ]:
df_eu_org.columns

In [ ]:
df_eu_proj.rename(columns={'id':'projectID'}, inplace=True)
df_eu_proj.columns

In [ ]:
df_eu_org['projectID'].nunique()
df_eu_proj['projectID'].nunique()

In [ ]:
df_eu_horizon_data= df_eu_org.merge(
    df_eu_proj,
    how='left', 
    on='projectID')

df_eu_horizon_data.head()

In [ ]:
df_eu_horizon_data.columns

### Target Projects - Focus on Health & Medtech investment

In [ ]:
df_keywords = (
    df_eu_horizon_data["keywords"]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
    .to_frame(name="keywords")
)

df_keywords.to_csv("data/Reports/unique_keywords.csv", index=False)

In [ ]:
keywords = [
    # Core Tecan business
    "laboratory automation", "lab automation", "automation",
    "liquid handling", "sample preparation", "workflow automation",
    "robotics", "robotic", "high-throughput", "screening",

    # Life Sciences focus
    "genomics", "proteomics", "multi-omics", "multiomics",
    "cell biology", "cellomics", "single cell", "cell and tissue",
    "organoids", "3d cell", "assay", "biomarker",

    # Diagnostics / clinical
    "diagnostics", "molecular diagnostics", "clinical diagnostics",
    "ivd", "in vitro diagnostics", "liquid biopsy",
    "genetic testing", "companion diagnostics",

    # Pharma / biotech
    "drug discovery", "drug development", "biopharma",
    "biotechnology", "personalized medicine", "precision medicine",
    "adme", "tox", "quality control",

    # Digital / AI relevance
    "artificial intelligence", "machine learning",
    "bioinformatics", "data-driven", "digital health",

    # Partnering / OEM / MedTech
    "medtech", "medical device", "oem", "cdmo",
    "contract manufacturing", "microfluidics"
]

In [ ]:
pattern = "|".join(keywords)

df_final = df_eu_horizon_data[

    df_eu_horizon_data["objective"]
    .fillna("")
    .str.lower()
    .str.contains(pattern, regex=True)
]

df_final.columns

## 2. CLINICAL TRIAL DATASETS

In [ ]:
url = "https://clinicaltrials.gov/api/v2/studies"

europe_countries = [
    "Germany", "France", "Switzerland", "United Kingdom", "Spain",
    "Italy", "Netherlands", "Belgium", "Sweden", "Denmark",
    "Austria", "Norway", "Finland", "Ireland", "Poland"
]

rows = []

for country in europe_countries:
    params = {
        "query.cond": "oncology OR genomics OR liquid biopsy",
        "query.locn": country,
        "filter.overallStatus": "RECRUITING,ACTIVE_NOT_RECRUITING,NOT_YET_RECRUITING",
        "pageSize": 100,
        "format": "json"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    for study in data.get("studies", []):
        protocol = study.get("protocolSection", {})
        identification = protocol.get("identificationModule", {})
        status = protocol.get("statusModule", {})
        design = protocol.get("designModule", {})
        sponsor = protocol.get("sponsorCollaboratorsModule", {})
        conditions = protocol.get("conditionsModule", {})
        dates = status.get("startDateStruct", {})

        rows.append({
            "country_search": country,
            "nct_id": identification.get("nctId"),
            "title": identification.get("briefTitle"),
            "status": status.get("overallStatus"),
            "start_date": dates.get("date"),
            "phase": design.get("phases", []),
            "sponsor": sponsor.get("leadSponsor", {}).get("name"),
            "conditions": conditions.get("conditions", [])
        })

df_trials = pd.DataFrame(rows)

print(df_trials.head())

In [ ]:
df_trials.columns

In [ ]:
df_trials['country_search'].unique()

### 3. US INFORMATION

In [ ]:
url2="https://api.reporter.nih.gov/v2/projects/search"

payload = {
    "criteria": {
        "advanced_text_search": {
            "operator": "and",
            "search_field": "all",
            "search_text": "genomics OR diagnostics OR oncology"
        },
        "fiscal_years": [2024, 2025]
    },
    "limit": 100,
    "offset": 0
}

response = requests.post(url2, json=payload)
data = response.json()
rows = []

for project in data["results"]:
    org = project.get("organization", {})
    rows.append({
        "project_title": project.get("project_title"),
        "fiscal_year": project.get("fiscal_year"),
        "award_amount": project.get("award_amount"),
        "institution": org.get("org_name"),
        "city": org.get("org_city"),
        "country": org.get("org_country"),
        "abstract": project.get("abstract_text")
    })

df_nih = pd.DataFrame(rows)

print(df_nih.head())


In [ ]:
df_nih.columns

In [ ]:
df_nih['country'].unique()

In [ ]:
df_nih['city'].unique()